In [ ]:
from google.colab import drive
drive.mount("/content/drive")

EXP  = "s05_readout_bias"
REPO = "/content/motionnet_cnn"
CKPT = f"/content/drive/MyDrive/NUIN/motionnet/ckpt/{EXP}"
LOG  = f"/content/drive/MyDrive/NUIN/motionnet/ckpt/{EXP}.log"

!if [ -d {REPO} ]; then cd {REPO} && git fetch -q origin && git checkout -q main && git pull -q origin main; \
 else git clone -q -b main https://github.com/guardomayas/motionnet.git {REPO}; fi
!pip install -q -e {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
import pandas as pd
from pathlib import Path
D = Path(CKPT)
runs = pd.read_csv(D / "summary.csv")
runs.groupby("model.readout_bias")[["nmse", "r2_x", "r2_y", "svar", "rate2"]].agg(["mean", "std"]).round(3)

In [ ]:
import torch, numpy as np
from motionnet.paths import setup, compose
from motionnet.dataset import get_data, batches
from motionnet.train.loop import load_checkpoint
from motionnet.models.cnn import unit

P = setup(True)
cfg, _ = compose(exp_name=EXP, corpus=P.corpus)
data = get_data(cfg["stimulus"], P.movies, "cuda")
vb = next(iter(batches(data.val, 32)))

rows = []
for f in sorted(D.glob("*.pt")):
    if f.stem.endswith("_last"):
        continue
    model, ck = load_checkpoint(f, "cuda", data.info)
    with torch.no_grad():
        model(vb["movie"].to("cuda"))
    R = unit(model.readout, (1,)).detach()
    w = R.pow(2).sum(0).cpu().numpy()
    act = model._active2.cpu().numpy()
    always_on = act > 0.95
    rows.append(dict(bias=ck["cfg"]["model"]["readout_bias"], seed=ck["cfg"]["optim"]["seed"],
                     n_always_on=int(always_on.sum()),
                     w_always_on=float(w[always_on].sum()),       # readout weight spent on them
                     bias_x=float(model.readout_bias[0]), bias_y=float(model.readout_bias[1])))
pd.DataFrame(rows).sort_values(["bias", "seed"])